# SafeSpace RAG System

## Step 1.1: Define the Knowledge Base

**Domain: self-help and mental wellness.** SafeSpace is a wellness app with private journals, community forums, friends, and self-care quests. Its users ask questions like "How can I sleep better?" or "I feel lonely, what can I do?". A RAG assistant for SafeSpace should answer from **trustworthy health sources** instead of from whatever the language model happens to remember, so the knowledge base is built from U.S. government health publications.

**Why these sources**
- **Trustworthy:** written by the National Institutes of Health (NIH) and reviewed by its scientists.
- **Free to reuse:** NIMH content is in the public domain. *NIH News in Health* states: "Our material is not copyrighted. Please acknowledge NIH News in Health as the source." Every document keeps its `Source:` line for credit.
- **Matches SafeSpace's features:** each topic maps to something users do in the app.

| # | Document | Publisher | Date | Words | SafeSpace connection |
|---|---|---|---|---|---|
| 1 | [Caring for Your Mental Health](https://www.nimh.nih.gov/health/topics/caring-for-your-mental-health) | NIMH | web page | 545 | self-care quests, when to get professional help |
| 2 | [Build Social Bonds to Protect Health](https://newsinhealth.nih.gov/2025/03/build-social-bonds-protect-health) | NIH News in Health | Mar 2025 | 1,158 | forums and friends |
| 3 | [Feeling Stressed? Ways to Improve Your Well-Being](https://newsinhealth.nih.gov/2021/01/feeling-stressed) | NIH News in Health | Jan 2021 | 1,088 | stress and relaxation quests |
| 4 | [Good Sleep for Good Health](https://newsinhealth.nih.gov/2021/04/good-sleep-good-health) | NIH News in Health | Apr 2021 | 1,089 | sleep habits |
| 5 | [Warning Signs of Suicide](https://www.nimh.nih.gov/health/publications/warning-signs-of-suicide) | NIMH (OM 25-4316) | rev. 2025 | 191 | safety: recognizing a crisis, 988 |
| 6 | [Practicing Gratitude](https://newsinhealth.nih.gov/2019/03/practicing-gratitude) | NIH News in Health | Mar 2019 | 601 | gratitude journaling |

**Total: 6 documents, 27 sections, 46 paragraphs, 4,672 words** (counted by the parser in Step 1.2).

**How the file was prepared**
- The article text was copied into one plain-text file, `knowledge_base.txt`. Images, captions, links and sidebars were left out.
- Bullet lists were rewritten as full sentences, so every paragraph reads as normal text and can be chunked by sentence.
- The text itself was not reworded. One factual update: the Stress article's old crisis line (1-800-273-TALK) was changed to the current **988** Suicide & Crisis Lifeline, which replaced it in 2022.

**Scope:** this assistant gives general wellness information. It is not a substitute for a therapist or doctor, and Step 1.5 keeps it from answering outside the knowledge base.

## Step 1.2: Text Representation & Chunking

**File format of `knowledge_base.txt`:**
- `# Document: <title>` starts a document; the next line is `Source: ...`.
- `## <heading>` starts a section, which is one topic.
- A **blank line ends a paragraph**.

### 1. Load and parse the knowledge base

**How we define a paragraph:** a block of consecutive non-empty lines between blank lines, inside a `##` section. Every paragraph keeps its document title and section heading as metadata. Later this lets each chunk point back to its source, and stops chunks from mixing topics.

In [1]:
import re
from collections import Counter

with open("knowledge_base.txt", encoding="utf-8") as f:
    raw = f.read()

def parse_knowledge_base(text):
    """Return a list of paragraph dicts: doc, source, section, text."""
    paragraphs = []
    doc = source = section = None
    buffer = []

    def flush():
        if buffer:
            paragraphs.append({"doc": doc, "source": source, "section": section,
                               "text": " ".join(buffer)})
            buffer.clear()

    for line in text.splitlines():
        line = line.strip()
        if line.startswith("# Document:"):
            flush()
            doc = line[len("# Document:"):].strip()
        elif line.startswith("Source:"):
            source = line[len("Source:"):].strip()
        elif line.startswith("## "):
            flush()
            section = line[3:].strip()
        elif line == "":
            flush()                      # blank line = end of paragraph
        else:
            buffer.append(line)
    flush()
    return paragraphs

paragraphs = parse_knowledge_base(raw)
for i, p in enumerate(paragraphs):
    p["id"] = i
    p["words"] = len(p["text"].split())

print(f"Documents : {len({p['doc'] for p in paragraphs})}")
print(f"Sections  : {len({(p['doc'], p['section']) for p in paragraphs})}")
print(f"Paragraphs: {len(paragraphs)}")
print(f"Words     : {sum(p['words'] for p in paragraphs)}")
print(f"Paragraph length: min {min(p['words'] for p in paragraphs)}, "
      f"max {max(p['words'] for p in paragraphs)}, "
      f"avg {sum(p['words'] for p in paragraphs) / len(paragraphs):.0f} words")

Documents : 6
Sections  : 27
Paragraphs: 46
Words     : 4672
Paragraph length: min 70, max 177, avg 102 words


In [2]:
import pandas as pd

pd.DataFrame(paragraphs).groupby("doc", sort=False).agg(
    paragraphs=("id", "count"), words=("words", "sum")
)

,paragraphs,words
doc,,
Caring for Your Mental Health,6,545
Build Social Bonds to Protect Health,11,1158
Feeling Stressed? Ways to Improve Your Well-Being,11,1088
Good Sleep for Good Health,11,1089
Warning Signs of Suicide,2,191
Practicing Gratitude,5,601


### 2. Sentence splitting

Fixed-length chunking must never cut a sentence in half, so we first split paragraphs into sentences. A sentence ends with `.`, `!` or `?` (possibly followed by a closing quote or bracket), then whitespace and a capital letter. The abbreviation **"Dr."** appears often in these articles, so it is protected and doesn't count as a sentence end.

In [3]:
def split_sentences(text):
    protected = text.replace("Dr.", "Dr<DOT>")
    # keep a closing quote with its own sentence instead of moving it to the next one
    protected = re.sub(r'([.!?]["”’)])\s+', r'\1\n', protected)
    protected = re.sub(r'(?<=[.!?])\s+(?=["“]?[A-Z0-9])', '\n', protected)
    return [s.replace("Dr<DOT>", "Dr.").strip() for s in protected.split("\n") if s.strip()]

example = paragraphs[6]["text"]
for s in split_sentences(example)[:5]:
    print("-", s)

- From an early age, we learn that nutritious foods and physical activity can help us stay healthy.
- Growing evidence now suggests that social connections may also be key to good health.
- Socially connected people tend to live longer.
- They're at lower risk for serious health problems.
- Social bonds are also linked to our mental health, eating habits, and much more.


### 3. Strategy 1: Fixed-length chunking

- **Target size:** about **150 words** per chunk.
- We add **whole sentences** until the next one would go past 150 words, then start a new chunk. Chunks are therefore *close to* 150 words but never cut a sentence.
- Chunks are built **inside one document** and never cross into another article, so a chunk never mixes unrelated topics (for example sleep and gratitude). Inside an article, a chunk may run across a paragraph break or a `##` heading. That is what makes it *fixed-length*: the size rule decides where a chunk ends, not the text's structure.

In [4]:
def group_by(paragraphs, key):
    """Group paragraphs by a key function, keeping file order."""
    groups = {}
    for p in paragraphs:
        groups.setdefault(key(p), []).append(p)
    return groups

def make_chunk(doc, sentences_or_paras, sections, paragraph_ids):
    return {"doc": doc, "sections": list(dict.fromkeys(sections)),
            "paragraph_ids": list(dict.fromkeys(paragraph_ids)),
            "text": " ".join(sentences_or_paras)}

def fixed_length_chunks(paragraphs, max_words=150):
    chunks = []
    for doc, paras in group_by(paragraphs, lambda p: p["doc"]).items():
        # (sentence, section, paragraph id) for every sentence in the document
        sentences = [(s, p["section"], p["id"]) for p in paras for s in split_sentences(p["text"])]
        current = []
        for item in sentences:
            n_current = sum(len(s.split()) for s, _, _ in current)
            if current and n_current + len(item[0].split()) > max_words:
                chunks.append(make_chunk(doc, *zip(*current)))
                current = []
            current.append(item)
        if current:
            chunks.append(make_chunk(doc, *zip(*current)))
    return chunks

fixed_chunks = fixed_length_chunks(paragraphs)
print(len(fixed_chunks), "fixed-length chunks")

36 fixed-length chunks


### 4. Strategy 2: Overlapping paragraph chunking

**How overlap is implemented:** a sliding window of **2 paragraphs** that moves **1 paragraph** at a time (stride 1), so neighbouring chunks share a whole paragraph (50% overlap):

- Chunk 1 = P1 + P2
- Chunk 2 = P2 + P3
- Chunk 3 = P3 + P4

The window slides **inside one document** and restarts at the next document, so it never joins two unrelated articles. A document with only 2 paragraphs (Warning Signs of Suicide) becomes one chunk. If a fact sits near the end of P2, it is retrieved together with the context before it (chunk 1) *and* after it (chunk 2).

In [5]:
def overlapping_chunks(paragraphs, window=2, stride=1):
    chunks = []
    for doc, paras in group_by(paragraphs, lambda p: p["doc"]).items():
        starts = range(0, max(len(paras) - window, 0) + 1, stride)
        for start in starts:
            group = paras[start:start + window]
            chunks.append(make_chunk(doc, [p["text"] for p in group],
                                     [p["section"] for p in group],
                                     [p["id"] for p in group]))
    return chunks

overlap_chunks = overlapping_chunks(paragraphs)
print(len(overlap_chunks), "overlapping chunks")

40 overlapping chunks


### 5. Strategy 3: Hybrid / strategic (section-based) chunking

Our own strategy follows the **structure the authors gave the articles**:

1. **One `##` section = one chunk.** Each section is one topic (for example "Just breathe" or "Sleep myths and truths"), so every chunk is one complete thought.
2. **Size cap:** if a section is longer than **250 words**, split it **only at paragraph breaks**, packing paragraphs while they fit. This keeps chunks small enough for the embedding models (MiniLM and MPNet read about 256 to 384 tokens).
3. **Context prefix:** start each chunk with `Document title > Section heading`. Short chunks such as "Tips for a better night's sleep" then still say what they are about, which helps both embedding search and the final answer's citations.

In [6]:
def hybrid_chunks(paragraphs, max_words=250):
    chunks = []
    for (doc, section), paras in group_by(paragraphs, lambda p: (p["doc"], p["section"])).items():
        groups, current = [], []
        for p in paras:
            if current and sum(x["words"] for x in current) + p["words"] > max_words:
                groups.append(current)
                current = []
            current.append(p)
        groups.append(current)
        for group in groups:
            body = " ".join(p["text"] for p in group)
            chunks.append({"doc": doc, "sections": [section],
                           "paragraph_ids": [p["id"] for p in group],
                           "text": f"{doc} > {section}\n{body}"})
    return chunks

hybrid = hybrid_chunks(paragraphs)
print(len(hybrid), "hybrid chunks")

32 hybrid chunks


### 6. Compare the three strategies

In [7]:
strategies = {"fixed_length": fixed_chunks, "overlapping": overlap_chunks, "hybrid": hybrid}

rows = []
for name, chunks in strategies.items():
    lengths = [len(c["text"].split()) for c in chunks]
    rows.append({"strategy": name, "chunks": len(chunks),
                 "avg_words": round(sum(lengths) / len(lengths)),
                 "min_words": min(lengths), "max_words": max(lengths),
                 "total_words": sum(lengths)})
summary = pd.DataFrame(rows).set_index("strategy")
summary

,chunks,avg_words,min_words,max_words,total_words
strategy,,,,,
fixed_length,36,130,42,149,4672
overlapping,40,204,148,303,8158
hybrid,32,156,82,256,4987


`total_words` for the overlapping strategy is much larger than the corpus (4,672 words) because every inner paragraph is stored twice. That repetition is the overlap. The hybrid total is slightly larger because of the heading prefix (the 250-word cap applies to the body, not the prefix).

### 7. Example chunks

In [8]:
def show(chunk, label):
    words = len(chunk["text"].split())
    print(f"--- {label} | {chunk['doc']} > {' / '.join(chunk['sections'])} | {words} words")
    print(chunk["text"], "\n")

show(fixed_chunks[1], "Fixed-length chunk")

# Two neighbouring overlapping chunks from the Social Bonds article
social = [c for c in overlap_chunks if c["doc"] == "Build Social Bonds to Protect Health"][2:]
show(social[0], f"Overlap chunk, paragraphs {social[0]['paragraph_ids']}")
show(social[1], f"Overlap chunk, paragraphs {social[1]['paragraph_ids']}")

show(next(c for c in hybrid if c["sections"] == ["Just breathe"]), "Hybrid chunk")

--- Fixed-length chunk | Caring for Your Mental Health > How can I take care of my mental health? | 142 words
A balanced diet and plenty of water can improve your energy and focus throughout the day. Pay attention to your intake of caffeine and alcohol and how they affect your mood and well-being—for some, decreasing caffeine and alcohol consumption can be helpful. Make sleep a priority. Stick to a schedule, and make sure you're getting enough sleep. Blue light from devices and screens can make it harder to fall asleep, so reduce blue light exposure from your phone or computer before bedtime. Try a relaxing activity. Explore relaxation or wellness programs or apps, which may incorporate meditation, muscle relaxation, or breathing exercises. Schedule regular times for these and other healthy activities you enjoy, such as listening to music, reading, spending time in nature, and engaging in low-stress hobbies. Set goals and priorities. Decide what must get done now and what can wait. 

-

### 8. Automatic rule checks

We check the assignment's three rules in code, not only by eye:

1. **No split sentences:** every chunk ends with sentence punctuation, and every sentence in the corpus appears whole inside at least one chunk.
2. **No merged topics:** every chunk comes from exactly one document, and every hybrid chunk from exactly one `##` section.
3. **Overlap works:** each pair of consecutive overlapping chunks in the same document shares exactly one paragraph.

In [9]:
all_sentences = [s for p in paragraphs for s in split_sentences(p["text"])]

for name, chunks in strategies.items():
    ends_ok = all(re.search(r'[.!?]["”’)]?$', c["text"]) for c in chunks)
    joined = "\n".join(c["text"] for c in chunks)
    whole_ok = all(s in joined for s in all_sentences)
    one_doc = all(len({paragraphs[i]["doc"] for i in c["paragraph_ids"]}) == 1 for c in chunks)
    print(f"{name:13s} ends with punctuation: {ends_ok} | all sentences intact: {whole_ok} "
          f"| one document per chunk: {one_doc}")

print("hybrid: one section per chunk:", all(len(c["sections"]) == 1 for c in hybrid))

pairs = [(a, b) for a, b in zip(overlap_chunks, overlap_chunks[1:]) if a["doc"] == b["doc"]]
shared = all(len(set(a["paragraph_ids"]) & set(b["paragraph_ids"])) == 1 for a, b in pairs)
print(f"\nOverlap: {len(pairs)} neighbouring pairs, each shares exactly one paragraph: {shared}")

fixed_length  ends with punctuation: True | all sentences intact: True | one document per chunk: True
overlapping   ends with punctuation: True | all sentences intact: True | one document per chunk: True
hybrid        ends with punctuation: True | all sentences intact: True | one document per chunk: True
hybrid: one section per chunk: True

Overlap: 34 neighbouring pairs, each shares exactly one paragraph: True


### Summary

| Strategy | Unit | Size rule | Overlap | Topic boundary |
|---|---|---|---|---|
| Fixed-length | sentences | about 150 words, never splits a sentence | none | stays inside one document |
| Overlapping paragraphs | paragraphs | 2 paragraphs per chunk | 1 paragraph (stride 1, 50%) | window restarts at each document |
| Hybrid / strategic | sections | 1 section, split at paragraph breaks if over 250 words | none | one section per chunk, with heading prefix |

## Step 1.3: Embedding Pipeline

### The three embedding models

| Size | Model | Dimensions | Parameters |
|---|---|---|---|
| Small | `BAAI/bge-small-en-v1.5` | 384 | 33M |
| Medium | `BAAI/bge-base-en-v1.5` | 768 | 110M |
| Large | `BAAI/bge-large-en-v1.5` | 1024 | 335M |

**Why one family:**
1. **Fair comparison.** All three share the same training recipe and data, so any difference in retrieval quality comes from model size rather than from different training.
2. **No truncation.** All three read up to **512 tokens**. Our longest chunks (about 300 words, roughly 400 tokens) fit completely. Popular alternatives such as `all-MiniLM-L6-v2` (256 tokens) and `all-mpnet-base-v2` (384 tokens) would silently cut the end off our longest chunks. We verify this in code below.

In [10]:
import time
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

device = "mps" if torch.backends.mps.is_available() else "cpu"   # Apple GPU if available
print("Device:", device)

MODEL_NAMES = {
    "small":  "BAAI/bge-small-en-v1.5",
    "medium": "BAAI/bge-base-en-v1.5",
    "large":  "BAAI/bge-large-en-v1.5",
}

models = {size: SentenceTransformer(name, device=device) for size, name in MODEL_NAMES.items()}

pd.DataFrame([{
    "size": size,
    "model": MODEL_NAMES[size],
    "dimensions": m.get_sentence_embedding_dimension(),
    "parameters (M)": round(sum(p.numel() for p in m.parameters()) / 1e6),
    "max tokens": m.max_seq_length,
} for size, m in models.items()]).set_index("size")

/Users/tungyuchen/PycharmProjects/9_SafeSpace/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1031 14:46:07.464000 47303 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Device: mps


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 12598.31it/s]
/var/folders/hq/4zsyf1dd77d29jc_2rbxgg9w0000gn/T/ipykernel_47303/2587736921.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "dimensions": m.get_sentence_embedding_dimension(),


,model,dimensions,parameters (M),max tokens
size,,,,
small,BAAI/bge-small-en-v1.5,384,33,512
medium,BAAI/bge-base-en-v1.5,768,109,512
large,BAAI/bge-large-en-v1.5,1024,335,512


### Check: does every chunk fit in 512 tokens?

Models read **tokens** (word pieces), not words. We count the tokens in every chunk with the model's own tokenizer. All three BGE models share the same tokenizer, so one check covers all of them.

In [11]:
tokenizer = models["small"].tokenizer
limit = models["small"].max_seq_length

for name, chunks in strategies.items():
    n_tokens = [len(tokenizer(c["text"])["input_ids"]) for c in chunks]
    print(f"{name:13s} longest chunk: {max(n_tokens)} tokens "
          f"(limit {limit}) -> truncated chunks: {sum(n > limit for n in n_tokens)}")

fixed_length  longest chunk: 206 tokens (limit 512) -> truncated chunks: 0
overlapping   longest chunk: 374 tokens (limit 512) -> truncated chunks: 0
hybrid        longest chunk: 327 tokens (limit 512) -> truncated chunks: 0


### The searchable structure: `VectorStore`

For each **model × chunking strategy** pair we build one `VectorStore`:

- `chunks`: the list of chunk dicts (text plus document and section metadata).
- `embeddings`: a NumPy matrix of shape **(number of chunks, dimensions)**. Row *i* is the vector for chunk *i*.

Each vector is **normalized to length 1** (`normalize_embeddings=True`). Then the **cosine similarity** between a query and every chunk is a single matrix product, `embeddings @ query_vector`. The `search` method returns the top-k most similar chunks and is used in Step 1.4.

With about 30–40 chunks per strategy, an exact NumPy search takes well under a millisecond, so a library such as FAISS isn't needed. The structure would work the same way with FAISS for a much larger knowledge base.

BGE recommends adding a short instruction in front of **queries only** (not chunks) for retrieval, so `search` adds it automatically.

In [12]:
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

class VectorStore:
    def __init__(self, model, chunks):
        self.model = model
        self.chunks = chunks
        self.embeddings = model.encode([c["text"] for c in chunks],
                                       normalize_embeddings=True, convert_to_numpy=True)

    def search(self, query, k=3):
        q = self.model.encode(BGE_QUERY_PREFIX + query, normalize_embeddings=True)
        scores = self.embeddings @ q                 # cosine similarity with every chunk
        top = np.argsort(scores)[::-1][:k]           # indices of the k highest scores
        return [(float(scores[i]), self.chunks[i]) for i in top]

### Embed every chunk: 3 models × 3 strategies = 9 vector stores

In [13]:
stores, rows = {}, []
for size, model in models.items():
    for strategy, chunks in strategies.items():
        start = time.perf_counter()
        stores[(size, strategy)] = VectorStore(model, chunks)
        rows.append({"model": size, "strategy": strategy,
                     "matrix shape": stores[(size, strategy)].embeddings.shape,
                     "seconds": round(time.perf_counter() - start, 2)})

pd.DataFrame(rows).set_index(["model", "strategy"])

matrix shape  seconds
model  strategy                          
small  fixed_length    (36, 384)     0.38
       overlapping     (40, 384)     0.33
       hybrid          (32, 384)     0.23
medium fixed_length    (36, 768)     0.50
       overlapping     (40, 768)     1.00
       hybrid          (32, 768)     0.76
large  fixed_length   (36, 1024)     1.70
       overlapping    (40, 1024)     3.53
       hybrid         (32, 1024)     2.82

### What an embedding looks like

Each chunk becomes a vector of 384, 768 or 1024 numbers. Here are the first 8 numbers of the first hybrid chunk, under each model. Each vector has length 1 because of normalization.

In [14]:
for size in models:
    vec = stores[(size, "hybrid")].embeddings[0]
    print(f"{size:6s} dim={len(vec):4d} length={np.linalg.norm(vec):.3f} first 8: {np.round(vec[:8], 3)}")

small  dim= 384 length=1.000 first 8: [ 0.036  0.002  0.065 -0.019  0.031  0.034  0.091 -0.02 ]
medium dim= 768 length=1.000 first 8: [-0.002  0.018 -0.008  0.017  0.053  0.052  0.08   0.003]
large  dim=1024 length=1.000 first 8: [-0.022 -0.02  -0.024  0.012 -0.035 -0.036  0.023  0.025]


## Step 1.4: Retrieval System

Retrieval runs in three steps:

1. **Query → embedding:** add BGE's search instruction in front of the query, encode it with the *same* model that embedded the chunks, and normalize it to length 1.
2. **Similarity search:** compute the cosine similarity between the query vector and every chunk vector. All vectors have length 1, so this is one matrix product: `embeddings @ query_vector`.
3. **Top-k retrieval:** sort the scores and keep the **k = 3** highest-scoring chunks.

First we walk through the steps one at a time for a single query.

In [15]:
query = "I've been having trouble sleeping. What habits can help me sleep better?"
store = stores[("large", "hybrid")]

# 1. Query -> embedding
query_vec = store.model.encode(BGE_QUERY_PREFIX + query, normalize_embeddings=True)
print("1. Query vector shape:", query_vec.shape)

# 2. Similarity search: cosine similarity with every chunk
scores = store.embeddings @ query_vec
print("2. One score per chunk:", scores.shape, "| range", round(scores.min(), 3), "to", round(scores.max(), 3))

# 3. Top-k retrieval
k = 3
top_k = np.argsort(scores)[::-1][:k]
print("3. Top-3 chunk indices:", top_k, "\n")
for rank, i in enumerate(top_k, 1):
    c = store.chunks[i]
    print(f"#{rank} score={scores[i]:.3f} | {c['doc']} > {' / '.join(c['sections'])}")

1. Query vector shape: (1024,)
2. One score per chunk: (32,) | range 0.444 to 0.694
3. Top-3 chunk indices: [24 25 23] 

#1 score=0.694 | Good Sleep for Good Health > Getting better sleep
#2 score=0.689 | Good Sleep for Good Health > Tips for a better night's sleep
#3 score=0.679 | Good Sleep for Good Health > Sleep disorders


### Test questions

Each question has the document(s) that *should* be retrieved, so retrieval can be checked automatically.

In [16]:
TEST_QUESTIONS = [
    {"topic": "Social",
     "query": "I feel lonely and want to connect with more people. How can I make more meaningful social connections?",
     "expected": {"Build Social Bonds to Protect Health"}},
    {"topic": "Relaxation",
     "query": "I'm feeling overwhelmed. Are there any relaxation tips?",
     "expected": {"Feeling Stressed? Ways to Improve Your Well-Being", "Caring for Your Mental Health"}},
    {"topic": "Sleep",
     "query": "I've been having trouble sleeping. What habits can help me sleep better?",
     "expected": {"Good Sleep for Good Health"}},
    {"topic": "Mental Health",
     "query": "I've been struggling with my mental health. Should I be seeking professional help, and where do I find it if so?",
     "expected": {"Caring for Your Mental Health"}},
    {"topic": "Self-Care",
     "query": "I want to build healthier daily habits. What are some self-care practices I can start with?",
     "expected": {"Caring for Your Mental Health"}},
]

### Compare all 9 vector stores (3 models × 3 chunking strategies)

For every store and question we record:
- **top-1 correct:** the highest-scoring chunk comes from an expected document;
- **hits in top 3:** how many of the 3 retrieved chunks come from an expected document;
- **top score:** the cosine similarity of the best chunk.

In [17]:
rows = []
for (size, strategy), store in stores.items():
    for q in TEST_QUESTIONS:
        results = store.search(q["query"], k=3)
        rows.append({"model": size, "strategy": strategy, "topic": q["topic"],
                     "top1_correct": results[0][1]["doc"] in q["expected"],
                     "hits_in_top3": sum(c["doc"] in q["expected"] for _, c in results),
                     "top_score": results[0][0]})
retrieval = pd.DataFrame(rows)

comparison = retrieval.groupby(["model", "strategy"], sort=False).agg(
    top1_correct=("top1_correct", "sum"),
    hits_in_top3=("hits_in_top3", "sum"),
    avg_top_score=("top_score", "mean"),
).round(3)
comparison["top1_correct"] = comparison["top1_correct"].astype(str) + " / 5"
comparison["hits_in_top3"] = comparison["hits_in_top3"].astype(str) + " / 15"
comparison

top1_correct hits_in_top3  avg_top_score
model  strategy                                             
small  fixed_length        5 / 5      13 / 15          0.727
       overlapping         5 / 5      14 / 15          0.759
       hybrid              5 / 5      14 / 15          0.773
medium fixed_length        5 / 5      12 / 15          0.670
       overlapping         4 / 5      13 / 15          0.677
       hybrid              5 / 5      14 / 15          0.714
large  fixed_length        5 / 5      12 / 15          0.681
       overlapping         5 / 5      14 / 15          0.704
       hybrid              5 / 5      14 / 15          0.708

In [18]:
# Which store got each question right at rank 1?
retrieval.pivot_table(index="topic", columns=["model", "strategy"],
                      values="top1_correct", sort=False).astype(bool)

model                small                          medium                     \
strategy      fixed_length overlapping hybrid fixed_length overlapping hybrid   
topic                                                                           
Social                True        True   True         True        True   True   
Relaxation            True        True   True         True        True   True   
Sleep                 True        True   True         True       False   True   
Mental Health         True        True   True         True        True   True   
Self-Care             True        True   True         True        True   True   

model                large                     
strategy      fixed_length overlapping hybrid  
topic                                          
Social                True        True   True  
Relaxation            True        True   True  
Sleep                 True        True   True  
Mental Health         True        True   True  
Self-Care             True        True   True

### Top-3 results for every question (large model, hybrid chunks)

In [19]:
def show_results(query, store, k=3):
    print("Q:", query)
    for rank, (score, c) in enumerate(store.search(query, k), 1):
        body = c["text"].split("\n", 1)[-1]
        print(f"  #{rank} {score:.3f} | {c['doc']} > {' / '.join(c['sections'])}")
        print(f"      {body[:140]}...")
    print()

for q in TEST_QUESTIONS:
    show_results(q["query"], stores[("large", "hybrid")])

Q: I feel lonely and want to connect with more people. How can I make more meaningful social connections?
  #1 0.758 | Build Social Bonds to Protect Health > New connections
      "If you're feeling lonely or socially disconnected, it can feel intimidating to try to form new connections," Necka says. "High-quality conn...
  #2 0.719 | Build Social Bonds to Protect Health > Social bonds
      Necka notes that there's a difference between social isolation and loneliness, but the two are related. Social isolation means you have few ...
  #3 0.696 | Build Social Bonds to Protect Health > Social bonds
      Many factors can contribute to a person feeling lonely. These include the quality of your personal relationships, your community, and societ...

Q: I'm feeling overwhelmed. Are there any relaxation tips?
  #1 0.646 | Feeling Stressed? Ways to Improve Your Well-Being > Ways to manage stress
      Be observant and recognize signs of excessive stress. These include difficulty sleeping, bein

## Step 1.5: Generation Pipeline

### Generator: our A6 model

We generate answers with **`Qwen/Qwen2.5-0.5B-Instruct`**, the model our team chose in A6 (`ai_prototype.ipynb`) and the one the SafeSpace Django app uses. The A7 model (`tiny_transformer.pt`) is a character-level Transformer with a 27-character vocabulary and a 32-character context window. It can't read a prompt containing several retrieved paragraphs, so it isn't usable for RAG generation.

### Prompt structure

The prompt has three parts, combined with Qwen's chat template:

1. **System instruction:** role, rules (answer only from the context, admit when the context doesn't cover something), and a safety note pointing to 988.
2. **Retrieved context:** the top-3 chunks, numbered and labelled with their source.
3. **User query**, followed by a one-line reminder of the main rule. Small models pay most attention to the end of the prompt, so the reminder is repeated there.

**Relevance threshold:** if even the best chunk scores below **0.5** cosine similarity, the knowledge base doesn't cover the question. We then skip generation and return a fixed reply. On-topic test questions score about 0.6–0.76, while unrelated questions score below 0.4.

**Retrieval store used for generation:** large model + hybrid chunks. In our comparison it is among the best stores, and its chunks carry section headings, which makes sources easy to cite. The `store` argument can be changed to any of the 9 stores.

In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, dtype=torch.float32).to(device)
gen_model.eval()
print("Loaded", GEN_MODEL, "on", device)

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 224.33it/s]


Loaded Qwen/Qwen2.5-0.5B-Instruct on mps


In [21]:
SYSTEM_PROMPT = (
    "You are SafeSpace's supportive wellness assistant. "
    "Answer the user's question using ONLY the information in the context below. "
    "Give practical, specific suggestions in a warm, encouraging tone, in 3 to 6 sentences. "
    "If the context does not contain the answer, say that you don't have information on that topic. "
    "You are not a therapist and cannot diagnose. If the user mentions thoughts of self-harm "
    "or suicide, tell them to call or text 988 (Suicide & Crisis Lifeline)."
)

def build_messages(query, results):
    context = "\n\n".join(
        f"[{i}] Source: {c['doc']} > {' / '.join(c['sections'])}\n{c['text'].split(chr(10), 1)[-1]}"
        for i, (_, c) in enumerate(results, 1)
    )
    user = (f"Context:\n{context}\n\nQuestion: {query}\n\n"
            "Answer using only the context above, in 3 to 6 sentences.")
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user}]

# Show one complete prompt, exactly as the model receives it
example_results = stores[("large", "hybrid")].search(TEST_QUESTIONS[2]["query"], k=3)
print(gen_tokenizer.apply_chat_template(build_messages(TEST_QUESTIONS[2]["query"], example_results),
                                        tokenize=False, add_generation_prompt=True))

<|im_start|>system
You are SafeSpace's supportive wellness assistant. Answer the user's question using ONLY the information in the context below. Give practical, specific suggestions in a warm, encouraging tone, in 3 to 6 sentences. If the context does not contain the answer, say that you don't have information on that topic. You are not a therapist and cannot diagnose. If the user mentions thoughts of self-harm or suicide, tell them to call or text 988 (Suicide & Crisis Lifeline).<|im_end|>
<|im_start|>user
Context:
[1] Source: Good Sleep for Good Health > Getting better sleep
If you're having trouble sleeping, hearing how important it is may be frustrating. But simple things can improve your odds of a good night's sleep. Treatments are available for many common sleep disorders. Cognitive behavioral therapy can help many people with insomnia get better sleep. Medications can also help some people. Many people with sleep apnea benefit from using a device called a CPAP machine. These ma

In [22]:
def generate(messages, max_new_tokens=250):
    prompt = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                    temperature=None, top_p=None, top_k=None)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]     # drop the prompt, keep the answer
    return gen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

NO_ANSWER = ("I'm sorry, I don't have information on that topic. I can help with stress, sleep, "
             "self-care, social connection, gratitude, and finding mental health support.")

def rag_answer(query, store=stores[("large", "hybrid")], k=3, min_score=0.5):
    results = store.search(query, k)                    # Step 1.4: retrieve
    if results[0][0] < min_score:                       # nothing relevant -> don't let the model guess
        answer = NO_ANSWER
    else:
        answer = generate(build_messages(query, results))   # Step 1.5: generate
    sources = [f"{c['doc']} > {' / '.join(c['sections'])} ({score:.2f})" for score, c in results]
    return answer, sources

### RAG answers to the test questions

Decoding is greedy (`do_sample=False`), so the answers are reproducible.

In [23]:
import textwrap

for q in TEST_QUESTIONS:
    answer, sources = rag_answer(q["query"])
    print(f"=== {q['topic']} ===")
    print("Q:", q["query"])
    print("A:", textwrap.fill(answer, 110, subsequent_indent="   "))
    print("Sources:")
    for s in sources:
        print("  -", s)
    print()

=== Social ===
Q: I feel lonely and want to connect with more people. How can I make more meaningful social connections?
A: To make more meaningful social connections, start by checking in with friends and family regularly. Joining
   clubs or groups that align with your interests can also help you meet new people. Consider volunteering or
   helping out in your community. Engaging in activities that challenge you, such as hiking or learning a new
   skill, can also strengthen your social network. Lastly, remember to take care of yourself physically and
   mentally by getting enough sleep, eating well, and practicing mindfulness.
Sources:
  - Build Social Bonds to Protect Health > New connections (0.76)
  - Build Social Bonds to Protect Health > Social bonds (0.72)
  - Build Social Bonds to Protect Health > Social bonds (0.70)

=== Relaxation ===
Q: I'm feeling overwhelmed. Are there any relaxation tips?
A: Relaxation tips include practicing controlled breathing, which involves focusin

### Out-of-scope question

The knowledge base has nothing about cooking. A grounded assistant should say so instead of inventing an answer. We show the model's output **without** the relevance threshold too: the 0.5B model ignores the system instruction and writes a recipe anyway. That is why the threshold is needed.

In [24]:
off_topic = "What is a good recipe for pasta carbonara?"
answer, sources = rag_answer(off_topic)
print("Retrieved (best matches, all low scores):", sources, "\n")
print("WITH threshold:", answer, "\n")

unguarded, _ = rag_answer(off_topic, min_score=0.0)
print("WITHOUT threshold:", textwrap.fill(unguarded, 110)[:600], "...")

Retrieved (best matches, all low scores): ['Build Social Bonds to Protect Health > Ways to connect with others (0.39)', "Good Sleep for Good Health > Tips for a better night's sleep (0.37)", "Build Social Bonds to Protect Health > Who's at risk? (0.37)"] 

WITH threshold: I'm sorry, I don't have information on that topic. I can help with stress, sleep, self-care, social connection, gratitude, and finding mental health support. 

WITHOUT threshold: A great recipe for pasta carbonara is to use fresh eggs, garlic, pancetta, and parmesan cheese. Cook the eggs
until set, then add the pancetta and sauté until crispy. In a separate pan, melt butter over medium heat. Add
the garlic and cook until fragrant. Stir in the pancetta and parmesan cheese until melted and creamy. Pour the
sauce into the egg mixture and mix gently until everything is combined. Serve hot with a side of crusty bread. ...


### RAG vs. no RAG

The same model and question, with and without retrieved context. Without context the model answers from its general training data, with no sources to check it against.

In [25]:
q = TEST_QUESTIONS[2]["query"]
no_rag = generate([{"role": "system", "content": "You are SafeSpace's supportive wellness assistant."},
                   {"role": "user", "content": q}])
with_rag, sources = rag_answer(q)

print("Q:", q, "\n")
print("WITHOUT RAG:\n", textwrap.fill(no_rag, 110), "\n")
print("WITH RAG:\n", textwrap.fill(with_rag, 110))
print("Sources:", sources)

Q: I've been having trouble sleeping. What habits can help me sleep better? 

WITHOUT RAG:
 Certainly! Here are some tips that might help you improve your sleep quality:  1. **Establish a Routine**: Try
to go to bed and wake up at the same time every day, even on weekends. This helps regulate your body’s
internal clock.  2. **Create a Relaxing Environment**: Ensure your bedroom is quiet, dark, and cool. Use
blackout curtains or eye masks if needed. Consider using earplugs for noise pollution.  3. **Limit Exposure to
Screens Before Bedtime**: The blue light from screens can interfere with your body’s natural sleep-wake cycle.
Try turning off phones, tablets, and computers an hour before bedtime.  4. **Exercise Regularly**: Physical
activity can help reduce stress and improve sleep. Aim for at least 30 minutes of moderate exercise most days
of the week.  5. **Avoid Stimulants**: Avoid caffeine, nicotine, and alcohol close to bedtime as they can
disrupt your sleep patterns.  6. **Try Rela

## Results and observations

**Retrieval (Step 1.4)**
- 8 of the 9 stores retrieve a correct document at rank 1 for all 5 test questions (medium + overlapping gets 4 of 5), so the knowledge base and embeddings work.
- **Hybrid chunks** are tied for the most hits in the top 3 (14 of 15) for every model, and get the highest average similarity score for every model. Their heading prefix ("Good Sleep for Good Health > Tips for a better night's sleep") adds topic words that match the question.
- **Fixed-length chunks** score lowest. A chunk can start mid-topic or run across a heading, which blurs its meaning.
- **Bigger isn't automatically better here.** The small model already finds the right document; scores between models aren't directly comparable (each model has its own score scale). With a small, clearly separated knowledge base, all three sizes are good enough; larger models matter more when documents overlap in topic.

**Generation (Step 1.5)**
- With RAG, answers follow the retrieved sources: the sleep answer reuses the NIH tips (consistent sleep schedule, 30 minutes outdoors, avoiding nicotine and caffeine, limiting screens, a cool and quiet room), and the mental-health answer gives the NIMH 2-week symptom rule, then the primary care provider, NIMH and SAMHSA. Compared with the no-RAG answer, the RAG answer is shorter and every tip can be traced to a source.
- Limits of a 0.5B model: it sometimes adds general advice that isn't in the context (for example "gardening" in the self-care answer and "stretching" in the relaxation answer), and it ignored "say you don't know" for an off-topic question. Repeating the rule after the question made answers shorter and more grounded than our first version. The relevance threshold fixes the off-topic case at the retrieval step, before the model can make something up.


## Notes for Step 2

In [26]:
# Cell 1: run the 45 retrieval tests with response time
def run_retrieval_tests(stores, questions=TEST_QUESTIONS, k=3):
    for model in {s.model for s in stores.values()}:      # warm-up: the first call is slow (loading onto the GPU)
        model.encode("warm-up")
    rows = []
    for (size, strategy), store in stores.items():
        for q in questions:
            start = time.perf_counter()
            results = store.search(q["query"], k=k)       # query embedding + similarity search + top-k
            ms = (time.perf_counter() - start) * 1000
            rows.append({"model": size, "strategy": strategy, "topic": q["topic"],
                         "top1_doc": results[0][1]["doc"],
                         "top1_section": " / ".join(results[0][1]["sections"]),
                         "top1_correct": results[0][1]["doc"] in q["expected"],
                         "hits_in_top3": sum(c["doc"] in q["expected"] for _, c in results),
                         "top_score": round(results[0][0], 3),
                         "response_ms": round(ms, 1)})
    return pd.DataFrame(rows)

tests_full = run_retrieval_tests(stores)
print(len(tests_full), "tests")
pd.set_option("display.max_rows", 100)
tests_full

# Cell 2: summary per store
tests_full.groupby(["model", "strategy"], sort=False).agg(
    top1_correct=("top1_correct", "sum"), hits_in_top3=("hits_in_top3", "sum"),
    avg_top_score=("top_score", "mean"), avg_ms=("response_ms", "mean")).round(3)

# Cell 3: smaller vs. full dataset
def build_stores(paras):
    chunk_sets = {"fixed_length": fixed_length_chunks(paras),
                  "overlapping": overlapping_chunks(paras),
                  "hybrid": hybrid_chunks(paras)}
    return {(size, name): VectorStore(model, chunks)
            for size, model in models.items() for name, chunks in chunk_sets.items()}

# keep the first half of every document's paragraphs (all 6 topics stay, about half the words)
small_paragraphs = [p for doc, paras in group_by(paragraphs, lambda p: p["doc"]).items()
                    for p in paras[:max(1, len(paras) // 2)]]
print(len(small_paragraphs), "paragraphs,", sum(p["words"] for p in small_paragraphs), "words")

tests_small = run_retrieval_tests(build_stores(small_paragraphs))
tests_full["dataset"], tests_small["dataset"] = "full", "small"
both = pd.concat([tests_full, tests_small])
both.groupby(["dataset", "model", "strategy"], sort=False).agg(
    top1_correct=("top1_correct", "sum"), hits_in_top3=("hits_in_top3", "sum"),
    avg_top_score=("top_score", "mean"), avg_ms=("response_ms", "mean")).round(3)

45 tests
21 paragraphs, 1884 words


top1_correct  hits_in_top3  avg_top_score  avg_ms
dataset model  strategy                                                       
full    small  fixed_length             5            13          0.728   13.60
               overlapping              5            14          0.759   12.16
               hybrid                   5            14          0.773   20.34
        medium fixed_length             5            12          0.669   23.48
               overlapping              4            13          0.677   34.24
               hybrid                   5            14          0.714   20.12
        large  fixed_length             5            12          0.681   38.30
               overlapping              5            14          0.704   45.06
               hybrid                   5            14          0.708   37.98
small   small  fixed_length             4            10          0.685    6.84
               overlapping              5            11          0.693    6.14
               hybrid                   5            11          0.690    6.34
        medium fixed_length             5            10          0.649   10.04
               overlapping              5            12          0.658    9.68
               hybrid                   5            12          0.662    9.24
        large  fixed_length             4            12          0.661   23.16
               overlapping              5            12          0.660   23.28
               hybrid                   5            13          0.659   23.28